# 01 - Data Audit, Integrity Validation & Chronological Splitting
### Fraud Risk Intelligence & Detection Platform
This notebook walks through:
1. Loading the credit card fraud dataset.
2. Running deterministic schema and data integrity validations (`DataValidator`).
3. Class imbalance inspection (0.17% fraud rate).
4. Temporal sorting and leak-free chronological 70/15/15 train-validation-test splitting.
5. Verifying no temporal leakage across split horizons.


In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

# Add project root to sys.path
sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))

from src.data import DataValidator, DataLoader
print("Imports loaded successfully.")


## 1. Load Raw Dataset & Run Schema Validation


In [ ]:
raw_csv = "data/raw/creditcard.csv"
df = pd.read_csv(raw_csv)
print(f"Dataset Shape: {df.shape[0]:,} rows, {df.shape[1]} columns")
df.head()


In [ ]:
validator = DataValidator()
report = validator.validate(df)
print(f"Valid: {report.is_valid}")
print(f"Total Rows: {report.total_records:,}")
print(f"Fraud Count: {report.metrics.get('fraud_count', 0):,} ({report.metrics.get('fraud_rate', 0):.4%})")
print(f"Errors: {report.errors}")
print(f"Warnings: {report.warnings}")


## 2. Inspect Class Imbalance & Monetary Distribution


In [ ]:
fraud_counts = df['Class'].value_counts()
print(f"Non-Fraud (0): {fraud_counts[0]:,} ({fraud_counts[0]/len(df):.4%})")
print(f"Fraud (1):     {fraud_counts[1]:,} ({fraud_counts[1]/len(df):.4%})")
print(f"Imbalance Ratio: 1 fraud per {fraud_counts[0]//fraud_counts[1]} legitimate transactions")


In [ ]:
print("Transaction Amount Summary by Class:")
print(df.groupby('Class')['Amount'].describe().round(2))


## 3. Chronological 70/15/15 Splitting (Strict Leakage Prevention)
In streaming fraud detection, random K-Fold split introduces future lookahead bias.
A transaction at timestamp $T$ can only use information from $[0, T)$. We enforce chronological ordering.


In [ ]:
loader = DataLoader(raw_path=raw_csv)
split_summary = loader.process_and_save(
    output_dir="data/processed",
    train_ratio=0.70,
    val_ratio=0.15,
    test_ratio=0.15
)

for k, v in split_summary.items():
    print(f"  {k}: {v}")


## 4. Verify Split Boundaries & No Time Overlap


In [ ]:
train_df = pd.read_parquet("data/processed/train.parquet")
val_df = pd.read_parquet("data/processed/val.parquet")
test_df = pd.read_parquet("data/processed/test.parquet")

print(f"Train Time Range: [{train_df['Time'].min():.0f}, {train_df['Time'].max():.0f}] (Total: {len(train_df):,})")
print(f"Val   Time Range: [{val_df['Time'].min():.0f}, {val_df['Time'].max():.0f}] (Total: {len(val_df):,})")
print(f"Test  Time Range: [{test_df['Time'].min():.0f}, {test_df['Time'].max():.0f}] (Total: {len(test_df):,})")

assert train_df['Time'].max() <= val_df['Time'].min(), "Temporal Leakage between Train and Val!"
assert val_df['Time'].max() <= test_df['Time'].min(), "Temporal Leakage between Val and Test!"
print("Verified: Strict chronological boundary ordering without temporal overlap!")
